In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model 
load_dotenv()  # Load environment variables from .env file

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

model = init_chat_model("gpt-4.1") # initialize a chat model with the name "gpt-4"

In [ ]:
from langchain.tools import tool

@tool # tool annotation helps to define a function as a tool that can be used by the agent
def get_weather(city: str) -> str:
    """get the weather for the city""" #docstring for the function that takes a city as input and returns a string indicating the weather in that city
    return f"The weather in this {city} is sunny"

# 🌟🌟🌟
#binding tool is similar to adding a tool list [] when creating an agent with create_agent function. 
# However, binding a tool to a model allows the model to use the tool without creating an agent. 
# This is useful when you want to use a tool with a model without creating an agent.

# bind the get_weather tool to the model so that the model can use it to answer questions about the weather
model_with_tools = model.bind_tools([get_weather]) 

In [ ]:
response = model_with_tools.invoke("What is the weather in New York?")
print(response) # Print the response text from the model

for tool_call in response.tool_calls:
    print("....................................")
    print(f"Tool called: {tool_call['name']}"), # we can see the llm is smart enough to call the tool get_weather with the argument New York
    print (f"arguments: {tool_call['args']}")  # Print the name and arguments of each tool call made by the model

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 52, 'total_tokens': 67, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_5cc2530a36', 'id': 'chatcmpl-EgEWfL9o9a3Gd8w70w61gwkxLznXJ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a1a2e5-a2bb-7304-ad49-11318d416ec2-0' tool_calls=[{'name': 'get_weather', 'args': {'city': 'New York'}, 'id': 'call_Gi3uqesBVVcp7fEeerf337I4', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 52, 'output_tokens': 15, 'total_tokens': 67, 'input_token_details': {'audio': 0, 'cache_read':

### Tool Executuion in Loop


In [29]:
# 1. Start the conversation with the user's question.
messages = [
    {
        "role": "user",
        "content": "What is the weather in New York?",
    }
]

# 2. Send the question to the model.
#    The model decides whether it can answer directly
#    or needs to call a tool.
ai_msg = model_with_tools.invoke(messages)

# 3. Print the model's first response.
#    This may contain a tool call instead of normal text.
print(f"AI Message's tool Call attributes: {ai_msg.tool_calls}")

# 4. Save the model's response in the conversation history.
#    The model's tool request must be preserved.
messages.append(ai_msg)

# 5. Check every tool the model requested.
for tool_call in ai_msg.tool_calls:

    # 6. Execute the requested tool.
    #    For example, get_weather receives:
    #    {"city": "New York"}
    tool_result = get_weather.invoke(tool_call)

    # 7. Add the tool's result to the conversation history.
    messages.append(tool_result)

# 8. Send the complete conversation back to the model.
#    The model now has:
#    - The user's original question
#    - Its own tool request
#    - The weather tool's result
final_response = model_with_tools.invoke(messages)

# 9. Print the final natural-language answer.
print(f"Final Response: {final_response.content}")

AI Message's tool Call attributes: [{'name': 'get_weather', 'args': {'city': 'New York'}, 'id': 'call_QACh2NSWShD6YYkSU8SOFhr6', 'type': 'tool_call'}]
Final Response: The weather in New York is sunny.
